# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The model outputs a `decay_risk_score`. We convert this raw score into clear, human-readable actions and reason codes so editors know exactly *why* a page was flagged and *what* to do with it.

- **PRIORITY_REFRESH (`High_Traffic_Decay_Risk`):** High risk score + High historical traffic. These are the crown jewels that are slipping. Update the content immediately.
- **INVESTIGATE_SEO (`Early_Crash_Risk`):** High risk score + Young page (<100 days). The page is dying prematurely. Check technical SEO or search intent mismatch.
- **STANDARD_REFRESH (`General_Decay`):** High risk score + average traffic. Put in the backlog for routine updates.
- **NO_ACTION (`Stable_or_Low_Risk`):** Low risk score. Do not touch.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. Load Data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
features = ['content_age_days', 'word_count', 'impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'engagement_rate']
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# 2. Train the Model
rf_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])
rf_pipe.fit(df[features], df['is_declining_label'])
df['decay_risk_score'] = rf_pipe.predict_proba(df[features])[:, 1]

# 3. Assign Actions and Reason Codes
def assign_action(row):
    if row['decay_risk_score'] > 0.6 and row['impressions_90d'] > 10000:
        return 'PRIORITY_REFRESH', 'High_Traffic_Decay_Risk'
    elif row['decay_risk_score'] > 0.6 and row['content_age_days'] < 100:
        return 'INVESTIGATE_SEO', 'Early_Crash_Risk'
    elif row['decay_risk_score'] > 0.6:
        return 'STANDARD_REFRESH', 'General_Decay'
    else:
        return 'NO_ACTION', 'Stable_or_Low_Risk'

df[['action', 'reason_code']] = df.apply(assign_action, axis=1, result_type='expand')
ranked_playbook = df.sort_values('decay_risk_score', ascending=False)

print("Top 5 Recommended Actions:")
print(ranked_playbook[['client_id', 'content_id', 'decay_risk_score', 'action', 'reason_code']].head(5))

Top 5 Recommended Actions:
               client_id            content_id  decay_risk_score  \
26453  client_3fdba35f04  content_9b6df29f7889          0.745959   
14992  client_3fdba35f04  content_b005f46f2e4c          0.744023   
18559  client_3fdba35f04  content_0d9c0ed65840          0.743496   
6842   client_3fdba35f04  content_91fefd1726b3          0.742684   
83     client_3fdba35f04  content_9f96edc42b03          0.742516   

                 action    reason_code  
26453  STANDARD_REFRESH  General_Decay  
14992  STANDARD_REFRESH  General_Decay  
18559  STANDARD_REFRESH  General_Decay  
6842   STANDARD_REFRESH  General_Decay  
83     STANDARD_REFRESH  General_Decay  


## 2. Intended use and limits

- **Intended Use:** This queue acts as **decision-support** for Content Editors to prioritize their weekly roadmap. It highlights patterns observed in historical data.
- **Limits:** This model is strictly observational. It does NOT prove that age causes decay, nor can it guarantee that refreshing a page will recover the traffic (we have not run a causal controlled experiment). It will also confidently misfire on highly seasonal pages.

## 3. Human review + the no-go list

- **Human Review Rules:** An editor must review the actual URL before starting work. They must check if the page is Seasonal (e.g. "Christmas Sweaters" in February) or strictly Evergreen (e.g. "What is a noun?"). If it is, they should dismiss the model's recommendation.
- **The No-Go List:** This model must **never be automated**. We should never automatically overwrite, delete, or AI-generate new content directly to the CMS based on this score. Automating this risks destroying brand voice, breaking compliance on legal pages, and hallucinating incorrect information.

## 4. Monitoring / retrain triggers

- **Monitoring:** Track `Precision@K` over the next 90-day window on live data. Out of the top 50 pages we flagged, how many actually dropped in real life?
- **Retrain Triggers:**
  1. A major Google Core Algorithm Update occurs (this changes the rules of SEO decay overnight, instantly invalidating the model's historical assumptions).
  2. Out-of-sample Precision@50 drops below 40% for two consecutive months.

## 5. Exports for the paper

We export the ranked queue and a JSON file containing headline metrics. These are the "receipts" that the final research paper will build upon.

In [2]:
import json

os.makedirs('../../work/outputs', exist_ok=True)
output_cols = ['client_id', 'content_id', 'decay_risk_score', 'action', 'reason_code', 'impressions_90d', 'content_age_days']

# Export the Queue
csv_path = '../../work/outputs/action_playbook_queue.csv'
ranked_playbook[output_cols].to_csv(csv_path, index=False)
print(f"Exported Playbook Queue to {csv_path}")

# Export the Metrics Receipt
metrics = {
    "total_pages_scored": len(ranked_playbook),
    "priority_refreshes_flagged": int(len(ranked_playbook[ranked_playbook['action'] == 'PRIORITY_REFRESH'])),
    "investigations_flagged": int(len(ranked_playbook[ranked_playbook['action'] == 'INVESTIGATE_SEO'])),
    "average_risk_score": float(ranked_playbook['decay_risk_score'].mean())
}
with open('../../work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics, f)
print("Exported playbook_metrics.json")

Exported Playbook Queue to ../../work/outputs/action_playbook_queue.csv
Exported playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.